# Reproducible transaction cleaning

Run all cells in a fresh kernel using the repository's Python environment. See `docs/data_cleaning.md` for the exact command and dependency versions.

The complete function below was recovered from the current working notebook's final cell and checked against commit `3f22682`. Commit `f0fd5f4` calls it without a saved definition. This replaces the exploratory cells, repeated exports and session-dependent recovery.

`delivered_usd` values the **recipient-received** amount using the reference price for the containing UTC transaction hour and the exact issuer/currency asset. It is not actual sender spending; conversion legs are not summed, and `SendMax` is only a ceiling. Raw payment fields and original transfers remain available downstream. Calendar and transfer count features are retained. No model training or changes to the team's 80/20 model split occur here.

In [ ]:
from pathlib import Path
import sys
import json
import importlib.metadata as metadata
import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/01_data_cleaning.ipynb").is_file()
             and (p / "data/raw").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Run from the repository root or a directory inside it")
TRAIN, VALIDATION, TEST = [ROOT / "data/raw" / split for split in ("train", "validation", "test")]
print("Working directory:", Path.cwd())
print("Repository:", ROOT)
print("Python:", sys.version, sys.executable)
print({p: metadata.version(p) for p in ("pandas", "numpy", "pyarrow", "nbformat", "nbclient", "ipykernel")})

LABEL_METADATA = {"family", "difficulty", "scenario_id", "cohort_id"}
NUMERIC_COLUMNS = ["LEDGER_INDEX", "TIMESTAMP", "TRANSACTION_INDEX", "SOURCE_FEE", "SEQUENCE", "FLAGS", "LAST_LEDGER_SEQUENCE"]

def validate_hashes(frame, context):
    hashes = frame["TRANSACTION_HASH"]
    if hashes.isna().any() or hashes.astype(str).str.strip().eq("").any() or not hashes.is_unique:
        raise ValueError(f"{context}: missing, blank or duplicate transaction hashes")


In [ ]:
from pathlib import Path
import pandas as pd
import json

OUTPUT_DIR = ROOT / "data/processed"

DROP_COLS = [
    "MEMOS", "TXN_SIGNATURE", "TICKET_SEQUENCE",
    "SIGNING_PUB_KEY", "SOURCE_TAG",
    "ACCOUNT_TRANSACTION_ID", "SIGNERS"
]

def process_dataset(folder, include_labels=False):
    tx = pd.read_parquet(folder / "transactions.parquet")
    tr = pd.read_parquet(folder / "transfers.parquet")
    exchange = pd.read_csv(folder / "market_rates.csv")

    validate_hashes(tx, f"{folder.name}: input transactions")
    input_hashes = set(tx["TRANSACTION_HASH"])
    if tr["TRANSACTION_HASH"].isna().any() or not set(tr["TRANSACTION_HASH"]).issubset(input_hashes):
        raise ValueError(f"{folder.name}: null or orphan transfer transaction hashes")
    if tr["TRANSFER_INDEX"].isna().any() or tr.duplicated(["TRANSACTION_HASH", "TRANSFER_INDEX"]).any():
        raise ValueError(f"{folder.name}: invalid transfer keys")
    # Preserve fields if a future dataset supplies values in them.
    tx = tx.drop(columns=[c for c in DROP_COLS if c in tx and tx[c].isna().all()])
    tx = tx.drop(columns=list(LABEL_METADATA | {"is_suspicious"}), errors="ignore")
    for col in NUMERIC_COLUMNS:
        tx[col] = pd.to_numeric(tx[col], errors="raise")
        if not np.isfinite(tx[col].to_numpy(dtype=float)).all():
            raise ValueError(f"{folder.name}: invalid numeric field {col}")

    # Aggregate transfers by transaction
    agg = tr.groupby("TRANSACTION_HASH").agg(
        transfer_count=("TRANSFER_INDEX", "size"),
        unique_senders=("FROM_ADDRESS", "nunique"),
        unique_receivers=("TO_ADDRESS", "nunique"),
        unique_tokens=("SYMBOL", "nunique")
    ).reset_index()

    tx = tx.merge(
        agg, on="TRANSACTION_HASH",
        how="left", validate="one_to_one"
    )

    count_cols = [
        "transfer_count", "unique_senders",
        "unique_receivers", "unique_tokens"
    ]
    tx[count_cols] = tx[count_cols].fillna(0).astype(int)

    # Temporal features
    dt = pd.to_datetime(tx["DATETIME"], utc=True)
    expected_dt = pd.to_datetime(tx["TIMESTAMP"], unit="s", utc=True)
    if dt.isna().any() or not dt.eq(expected_dt).all():
        raise ValueError(f"{folder.name}: DATETIME and TIMESTAMP disagree")
    tx["hour"] = dt.dt.hour
    tx["day_of_week"] = dt.dt.day_of_week
    tx["is_weekend"] = (dt.dt.day_of_week >= 5).astype(int)

    # Extract payment information
    def parse_amount(raw):
        value = json.loads(raw)
        if isinstance(value, dict):
            return (
                float(value["value"]),
                value["currency"],
                f'{value["issuer"]}:{value["currency"]}'
            )
        return float(value) / 1_000_000, "XRP", "XRP"

    parsed = tx["DELIVERED_AMOUNT"].map(parse_amount)

    tx["delivered_value"] = parsed.map(lambda x: x[0])
    tx["delivered_currency"] = parsed.map(lambda x: x[1])
    tx["asset"] = parsed.map(lambda x: x[2])
    tx["is_xrp_payment"] = (
        tx["delivered_currency"] == "XRP"
    ).astype(int)

    extra = tx["EXTRA_TRANSACTION_FIELDS"].map(json.loads)
    tx["has_sendmax"] = extra.map(
        lambda x: int("SendMax" in x)
    )
    tx["has_paths"] = extra.map(
        lambda x: int(bool(x.get("Paths")))
    )

    # Exact containing-hour reference price, keyed by issuer AND currency.
    tx["rate_hour"] = (tx["TIMESTAMP"] // 3600 * 3600).astype("int64")
    exchange["TIMESTAMP"] = pd.to_numeric(exchange["TIMESTAMP"], errors="raise")
    exchange["usd_price"] = pd.to_numeric(exchange["PRICE_MICRO_USD"], errors="raise") / 1_000_000
    if (exchange["TIMESTAMP"].isna().any()
            or (exchange["TIMESTAMP"] % 3600 != 0).any()
            or exchange["ASSET"].isna().any()
            or exchange.duplicated(["TIMESTAMP", "ASSET"]).any()
            or not np.isfinite(exchange["usd_price"]).all()
            or (exchange["usd_price"] <= 0).any()):
        raise ValueError(f"{folder.name}: invalid or duplicate hourly exchange rates")
    tx = tx.merge(
        exchange[["TIMESTAMP", "ASSET", "usd_price"]].rename(columns={"TIMESTAMP": "rate_hour"}),
        left_on=["rate_hour", "asset"], right_on=["rate_hour", "ASSET"],
        how="left", validate="many_to_one", sort=False,
    )
    missing = tx["usd_price"].isna()
    if missing.any():
        examples = tx.loc[missing, ["TRANSACTION_HASH", "rate_hour", "asset"]].head().to_dict("records")
        raise ValueError(f"{folder.name}: missing containing-hour rates for {missing.sum()} transactions: {examples}")
    if not np.isfinite(tx["delivered_value"]).all() or (tx["delivered_value"] < 0).any():
        raise ValueError(f"{folder.name}: invalid delivered amounts")
    # Recipient-received value only: never sum transfer legs or use SendMax.
    tx["delivered_usd"] = tx["delivered_value"] * tx["usd_price"]
    if not np.isfinite(tx["delivered_usd"]).all():
        raise ValueError(f"{folder.name}: invalid delivered USD")
    tx = tx.drop(columns="rate_hour")

    # Add target only for training
    if include_labels:
        lbl = pd.read_csv(
            folder / "labels.csv",
            usecols=["TRANSACTION_HASH", "is_suspicious"]
        )
        validate_hashes(lbl, f"{folder.name}: labels")
        if set(lbl["TRANSACTION_HASH"]) != input_hashes:
            raise ValueError(f"{folder.name}: labels do not cover exactly the input hashes")
        if lbl["is_suspicious"].isna().any() or not lbl["is_suspicious"].isin([0, 1]).all():
            raise ValueError(f"{folder.name}: missing or invalid targets")
        tx = tx.merge(
            lbl, on="TRANSACTION_HASH",
            how="left", validate="one_to_one"
        )
        assert tx["is_suspicious"].notna().all()

    validate_hashes(tx, f"{folder.name}: cleaned transactions")
    if len(tx) != len(input_hashes) or set(tx["TRANSACTION_HASH"]) != input_hashes:
        raise ValueError(f"{folder.name}: input/output hash coverage differs")
    return tx, tr


In [ ]:
# Process and validate every split before writing any final output.
datasets = {split: process_dataset(folder, include_labels=(split == "train"))
            for split, folder in [("train", TRAIN), ("validation", VALIDATION), ("test", TEST)]}
feature_columns = datasets["train"][0].drop(columns="is_suspicious").columns.tolist()
feature_types = datasets["train"][0][feature_columns].dtypes
for split, (tx, tr) in datasets.items():
    if tx.drop(columns="is_suspicious", errors="ignore").columns.tolist() != feature_columns:
        raise ValueError(f"{split}: feature columns or ordering differ")
    if not tx[feature_columns].dtypes.equals(feature_types):
        raise ValueError(f"{split}: feature types differ")
    if LABEL_METADATA.intersection(tx.columns) or (split != "train" and "is_suspicious" in tx):
        raise ValueError(f"{split}: label leakage")
    print(f"{split}: {len(tx):,} transactions, {len(tr):,} transfers; missing USD: {tx.delivered_usd.isna().sum()}")


In [ ]:
# Single final export step. Transfers are preserved unchanged.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for split, (tx, tr) in datasets.items():
    tx.to_parquet(OUTPUT_DIR / f"{split}_cleaned.parquet", index=False)
    tr.to_parquet(OUTPUT_DIR / f"{split}_transfers.parquet", index=False)

# Verify the actual files, including exact coverage and round-trip contents.
for split, (tx, tr) in datasets.items():
    saved_tx = pd.read_parquet(OUTPUT_DIR / f"{split}_cleaned.parquet")
    saved_tr = pd.read_parquet(OUTPUT_DIR / f"{split}_transfers.parquet")
    pd.testing.assert_frame_equal(saved_tx, tx)
    pd.testing.assert_frame_equal(saved_tr, tr)
    validate_hashes(saved_tx, f"{split}: exported transactions")
print("PASS: all six exports verified; complete hash coverage; consistent features; no label leakage.")
